# 02 — Compare Methods

You now have two distance functions. One is fast and wrong. One is correct.

This notebook puts them side by side — not just to show they differ, but to show *how much* they differ, *where* they differ most, and what drives the gap. By the end you should have a reliable instinct for when each approach is appropriate.

In [2]:
import math

def euclidean_distance(p1, p2):
    """Straight-line distance in degrees. Not a real distance."""
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def haversine_km(p1, p2):
    """Great-circle distance in kilometers."""
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))

## 1. Side-by-Side Comparison

The table below runs both methods across a range of pairs — from a few kilometers apart to intercontinental. The Euclidean column is in degrees; the Haversine column is in kilometers. They cannot be directly compared in value, but the pattern of divergence is clear.

In [3]:
pairs = [
    ("On-base (< 1 km)",        (-98.490, 33.910), (-98.495, 33.915)),
    ("Same city (~10 km)",       (-98.49, 33.91),   (-98.40, 33.98)),
    ("City to city (~130 km)",   (-98.5,  33.8),    (-97.2,  34.1)),
    ("WF → Dallas (~200 km)",    (-98.49, 33.91),   (-96.80, 32.78)),
    ("Dallas → Houston (~360 km)",(-96.80, 32.78),  (-95.37, 29.76)),
    ("Dallas → Denver (~1300 km)",(-96.80, 32.78),  (-104.99, 39.74)),
    ("Dallas → New York (~2200 km)",(-96.80, 32.78),(-74.01, 40.71)),
    ("Dallas → London (~8000 km)",(-96.80, 32.78),  (-0.13, 51.51)),
]

KM_PER_DEG = 111.32   # rough conversion at ~33°N for display only

print(f"{'Pair':<30}  {'Euclidean (°)':>14}  {'~km equiv':>10}  {'Haversine (km)':>15}")
print("-" * 76)
for label, a, b in pairs:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    approx = e * KM_PER_DEG
    print(f"{label:<30}  {e:>13.4f}°  {approx:>9.1f}   {h:>14.1f} km")

Pair                             Euclidean (°)   ~km equiv   Haversine (km)
----------------------------------------------------------------------------
On-base (< 1 km)                       0.0071°        0.8              0.7 km
Same city (~10 km)                     0.1140°       12.7             11.4 km
City to city (~130 km)                 1.3342°      148.5            124.5 km
WF → Dallas (~200 km)                  2.0330°      226.3            201.1 km
Dallas → Houston (~360 km)             3.3415°      372.0            362.3 km
Dallas → Denver (~1300 km)            10.7479°     1196.5           1065.9 km
Dallas → New York (~2200 km)          24.1303°     2686.2           2205.4 km
Dallas → London (~8000 km)            98.4678°    10961.4           7640.8 km


## 2. Error Growth

To compare the methods directly, we need a common unit. The Euclidean value in degrees can be scaled by `~111.32 km/°` (the rough latitude-to-km conversion) to get an approximate kilometer estimate — then compared against Haversine as the ground truth.

The percentage error shows how badly the scaling assumption breaks down at larger distances.

In [4]:
def pct_error(euclidean_deg, haversine_km_val):
    """
    Convert Euclidean degrees to a km estimate, compare to Haversine truth.
    Note: this conversion is approximate and latitude-dependent.
    """
    euclidean_approx_km = euclidean_deg * KM_PER_DEG
    return abs(euclidean_approx_km - haversine_km_val) / haversine_km_val * 100


print(f"{'Pair':<30}  {'Haversine':>12}  {'Eucl. approx':>13}  {'Error':>8}")
print("-" * 70)
for label, a, b in pairs:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    err = pct_error(e, h)
    flag = "  ← SIGNIFICANT" if err > 5 else ""
    print(f"{label:<30}  {h:>10.1f} km  {e * KM_PER_DEG:>11.1f} km  {err:>7.1f}%{flag}")

Pair                               Haversine   Eucl. approx     Error
----------------------------------------------------------------------
On-base (< 1 km)                       0.7 km          0.8 km      8.9%  ← SIGNIFICANT
Same city (~10 km)                    11.4 km         12.7 km     11.5%  ← SIGNIFICANT
City to city (~130 km)               124.5 km        148.5 km     19.3%  ← SIGNIFICANT
WF → Dallas (~200 km)                201.1 km        226.3 km     12.6%  ← SIGNIFICANT
Dallas → Houston (~360 km)           362.3 km        372.0 km      2.7%
Dallas → Denver (~1300 km)          1065.9 km       1196.5 km     12.2%  ← SIGNIFICANT
Dallas → New York (~2200 km)        2205.4 km       2686.2 km     21.8%  ← SIGNIFICANT
Dallas → London (~8000 km)          7640.8 km      10961.4 km     43.5%  ← SIGNIFICANT


The error is small at short range and grows as the distance increases. Two sources combine to produce it:

1. **Longitude compression** — the Euclidean formula uses a fixed `111.32 km/°` for both axes, but longitude degrees are shorter than latitude degrees at any latitude above 0°
2. **Curvature accumulation** — over long distances, the Earth's surface curves away from the flat-plane assumption, adding error that compounds with distance

These are not the same thing, and both are always present to some degree.

## 3. The Latitude Effect

Now isolate the longitude compression factor. Take one fixed degree of longitude separation — `Δlon = 1.0°` — and hold latitude constant per row. The Euclidean distance stays `1.0°` every time. The Haversine distance changes dramatically.

In [5]:
latitudes = [0, 10, 20, 30, 40, 50, 60, 70, 80]

print(f"{'Latitude':>10}  {'Euclidean (°)':>14}  {'Haversine (km)':>15}  {'Error':>8}")
print("-" * 56)
for lat in latitudes:
    a = (-10.0, float(lat))
    b = ( -9.0, float(lat))   # exactly 1° east, same latitude
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    err = pct_error(e, h)
    print(f"{lat:>9}°  {e:>13.4f}°  {h:>14.2f} km  {err:>7.1f}%")

  Latitude   Euclidean (°)   Haversine (km)     Error
--------------------------------------------------------
        0°         1.0000°          111.19 km      0.1%
       10°         1.0000°          109.51 km      1.7%
       20°         1.0000°          104.49 km      6.5%
       30°         1.0000°           96.30 km     15.6%
       40°         1.0000°           85.18 km     30.7%
       50°         1.0000°           71.47 km     55.7%
       60°         1.0000°           55.60 km    100.2%
       70°         1.0000°           38.03 km    192.7%
       80°         1.0000°           19.31 km    476.5%


The Euclidean value is identical in every row — `1.0°`. The Haversine value drops from ~111 km at the equator to ~19 km near 80°N. The error is zero at the equator (where the axes happen to be the same scale) and climbs steadily toward the poles.

This is not a rounding error or a precision issue. It is a geometric fact about what latitude and longitude mean on a sphere.

## 4. Visualize the Difference

Draw three lines on a single map: one short local pair, one medium pair, and one long pair. Each line is labeled with both distances — degree-based and kilometer-based — so the divergence is visible in the context of the actual geography.

In [6]:
from ipyleaflet import Map, GeoJSON

showcase = [
    ("Short",  (-98.49, 33.91), (-97.2,  34.1),  "#2a9d8f"),
    ("Medium", (-98.49, 33.91), (-96.80, 32.78),  "#e9c46a"),
    ("Long",   (-98.49, 33.91), (-74.01, 40.71),  "#e63946"),
]

features = []
for label, a, b, color in showcase:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(a), list(b)]},
        "properties": {
            "label": label,
            "euclidean_deg": round(e, 4),
            "haversine_km":  round(h, 1),
        }
    })
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": list(b)},
        "properties": {"name": label}
    })
    print(f"{label:<8}  Euclidean: {e:.4f}°   Haversine: {h:.1f} km")

# Add base point
features.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [-98.49, 33.91]},
    "properties": {"name": "Base"}
})

fc = {"type": "FeatureCollection", "features": features}

m = Map(center=(36.0, -88.0), zoom=4)
m.add(GeoJSON(
    data=fc,
    style={"color": "#457b9d", "weight": 2},
    hover_style={"color": "#e63946"}
))
m

Short     Euclidean: 1.3039°   Haversine: 120.8 km
Medium    Euclidean: 2.0330°   Haversine: 201.1 km
Long      Euclidean: 25.4069°   Haversine: 2284.2 km


Map(center=[36.0, -88.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

---

## Exercise A — Compute Percentage Error

For each pair below, compute the Euclidean distance in degrees, convert it to an approximate km estimate using `111.32 km/°`, and compute the percentage error against the Haversine result. Print a table sorted by error, largest to smallest.

In [7]:
exercise_pairs = [
    ("Tinker AFB → NAS Fort Worth",  (-97.37, 35.39), (-97.04, 32.85)),
    ("Dallas → Chicago",             (-96.80, 32.78), (-87.63, 41.88)),
    ("Dallas → Mexico City",         (-96.80, 32.78), (-99.13, 19.43)),
    ("OKC → Tulsa",                  (-97.52, 35.47), (-95.99, 36.15)),
    ("Dallas → Buenos Aires",        (-96.80, 32.78), (-58.38, -34.61)),
    ("Wichita Falls → Abilene",      (-98.49, 33.91), (-99.73, 32.45)),
]

def print_error_table(pairs):
    print(f"{'Pair':<30}  {'Haversine':>12}  {'Eucl. approx':>13}  {'Error':>8}")
    print("-" * 70)
    for label, a, b in pairs:
        e = euclidean_distance(a, b)
        h = haversine_km(a, b)
        err = pct_error(e, h)
        print(f"{label:<30}  {h:>10.1f} km  {e * KM_PER_DEG:>11.1f} km  {err:>7.1f}%")

print_error_table(exercise_pairs)


# your code here
# expected output: table sorted by error % descending

Pair                               Haversine   Eucl. approx     Error
----------------------------------------------------------------------
Tinker AFB → NAS Fort Worth          284.1 km        285.1 km      0.4%
Dallas → Chicago                    1295.0 km       1438.1 km     11.1%
Dallas → Mexico City                1502.4 km       1508.6 km      0.4%
OKC → Tulsa                          157.3 km        186.4 km     18.5%
Dallas → Buenos Aires               8498.7 km       8635.4 km      1.6%
Wichita Falls → Abilene              199.2 km        213.2 km      7.1%


## Exercise B — Find the Crossover Point

Starting from the base point `(-98.49, 33.91)`, step east in 0.5° increments and compute both distances for each step. Find the approximate separation (in km) where the Euclidean error first exceeds **5%**.

Print the crossover row.

In [9]:
base = (-98.49, 33.91)

def step_east(point, degrees):
    """Move east by a certain number of degrees (longitude)."""
    return (point[0] + degrees, point[1])

for step in range(1, 41):  # 0.5° increments up to 20°
    new_point = step_east(base, step * 0.5)
    e = euclidean_distance(base, new_point)
    h = haversine_km(base, new_point)
    err = pct_error(e, h)
    print(f"Step {step * 0.5:.1f}° east: Euclidean = {e:.4f}°, Haversine = {h:.2f} km, Error = {err:.1f}%")
    if err > 5:
        print("Error exceeded 5%. Stopping.")
        break

# your code here
# step east from base in 0.5° increments up to 20°
# for each step: compute euclidean_distance, haversine_km, pct_error
# stop and print the first row where error > 5%

Step 0.5° east: Euclidean = 0.5000°, Haversine = 46.14 km, Error = 20.6%
Error exceeded 5%. Stopping.


## Exercise C — Explain the Errors

Look at your Exercise A results. Answer the following in plain English (comments or print statements are fine):

1. Which pair had the highest error? Why — is it mostly about distance, latitude change, or both?
2. Which pair had the lowest error? What makes it a "safe" case for Euclidean approximation?
3. The Dallas → Mexico City and Dallas → Chicago pairs are at similar distances. Do they have similar errors? If not, why not?

In [ ]:
# 1. HIGHEST ERROR: OKC → Tulsa at 18.5%
#
#    Counterintuitively, this is a *short* trip — but it travels almost due east,
#    meaning most of the separation is in longitude degrees. 
#    Short trips also have no chance to average out the distortion across
#    a changing latitude, so the longitude compression hits in full.
#
#    Distance alone is not the culprit — direction is.

# 2. LOWEST ERROR: Tinker AFB → NAS Fort Worth at 0.4%
#    (tied with Dallas → Mexico City)
#
#    This pair travels mostly south-southwest — primarily a change in latitude
#    with a modest longitude component. Latitude degrees are constant (~111 km)
#    everywhere, so Euclidean handles north-south displacement accurately.
#    The small east-west component introduces a small longitude error, but
#    it barely moves the overall result. A "safe" Euclidean case is one where
#    the path is mostly north-south and spans a small, consistent latitude band.

# 3. DALLAS → CHICAGO vs DALLAS → MEXICO CITY: similar distance, very different error
#
#    Dallas → Chicago:      1295 km Haversine, 11.1% error
#    Dallas → Mexico City:  1502 km Haversine,  0.4% error
#
#    Chicago is almost due NORTH of Dallas — the path crosses from ~33°N to ~42°N,
#    gaining 9° of latitude. As latitude increases, longitude degrees shrink,
#    and the great-circle arc curves noticeably. Euclidean ignores both effects,
#    producing significant error on a long, high-latitude-crossing path.
#
#    Mexico City is almost due SOUTH — the path goes from ~33°N down to ~19°N,
#    staying in a warmer latitude band where the longitude compression is
#    more uniform, and the route is predominantly a latitude change (which
#    Euclidean handles well). The errors largely cancel rather than compound.
#
#    The lesson: it is the DIRECTION of travel and the LATITUDE BAND crossed,
#    not the distance alone, that determines how badly Euclidean degrades.



---

## Check Your Understanding

A teammate writes a function to find all points within a given radius and uses Euclidean distance to avoid importing anything:

```python
def within_radius(base, points, radius_km):
    threshold = radius_km / 111.32   # convert km to degrees
    return [p for p in points if euclidean_distance(base, p) <= threshold]
```

They test it in Texas on a small local dataset and it seems to work. Then the dataset expands to include points in Canada and the results are wrong — some points inside the radius are excluded, others outside are included.

**Question:** Identify two distinct reasons why this function produces incorrect results for a dataset spanning Texas to Canada. For each reason, state whether it would cause false positives (non-members returned), false negatives (members excluded), or both.

```python
# your answer here
```


---

## Next

In [03 — Distance Applications](./03-Distance_Applications.ipynb), we put `haversine_km` to work on real spatial problems: nearest-neighbor search, radius filtering, and distance-from-click interactions.